目標是學習使用bert接到自己的模型上

In [14]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "google/bert_uncased_L-2_H-128_A-2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
bert = AutoModel.from_pretrained(model_name)


Loading weights: 100%|██████████| 39/39 [00:00<00:00, 1223.69it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# 定義分類模型
class MyClassifier(nn.Module):
    def __init__(self, bert, num_classes):
        super().__init__()
        self.bert = bert
        hidden_size = bert.config.hidden_size
        
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        cls = outputs.last_hidden_state[:, 0, :]  # 取 CLS token
        
        logits = self.mlp(cls)
        return logits


In [17]:
from pathlib import Path
from torch.utils.data import Dataset
class FolderDataset(Dataset):
    def __init__(self, root_dir, split):
        root_dir = Path(root_dir)
        base = root_dir/ split
        self.samples = []
        for label_name, label in [("pos", 1), ("neg", 0)]:
            folder = base / label_name
            for fp in folder.glob("*.txt"):
                self.samples.append((fp, label))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self,index):
        value,label = self.samples[index]
        text = value.read_text()
        return text,label
train_ds = FolderDataset("data/aclImdb","train")
test_ds = FolderDataset("data/aclImdb","test")
print(len(train_ds))    
print(len(test_ds))

25000
25000


In [ ]:
def collate_fn(batch):
    texts = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch])
    
    encodings = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt" # 返回 PyTorch 張量
    )
    
    encodings["labels"] = labels
    return encodings


In [19]:
from torch.utils.data import DataLoader
train_loader = DataLoader(
    train_ds,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_ds,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn
)


In [21]:
num_classes = 1  # 二分類問題，輸出一個 logit

model = MyClassifier(bert, num_classes).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)


In [25]:
criterion = nn.BCEWithLogitsLoss()#二分類問題用這個損失函數，輸出一個 logit

n_batches = len(train_loader)
total_loss = 0.0
total_acc = 0.0

for batch in train_loader:
    input_ids = batch["input_ids"].to(device)
    mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device).float().unsqueeze(1)  # [B,1]

    logits = model(input_ids, mask)  # [B,1]
    loss = criterion(logits, labels)

    preds = (torch.sigmoid(logits) > 0.5).float()
    total_acc += (preds == labels).float().mean().item()  # .item()是將張量轉換為Python數值
    total_loss += loss.item()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(f"Loss: {total_loss / n_batches:.4f}, Accuracy: {total_acc / n_batches:.4f}")


Loss: 0.3225, Accuracy: 0.8645


version 1 Loss: 0.4706, Accuracy: 0.7718 12min
version 2 Loss: 0.3225, Accuracy: 0.8645 13min

In [26]:
# 評估模型
model.eval()
n_batches = len(test_loader)
total_loss = 0.0
total_acc = 0.0 
for batch in test_loader:
    input_ids = batch["input_ids"].to(device)
    mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device).float().unsqueeze(1)  # [B,1]

    with torch.no_grad():
        logits = model(input_ids, mask)  # [B,1]
        loss = criterion(logits, labels)

    preds = (torch.sigmoid(logits) > 0.5).float()
    total_acc += (preds == labels).float().mean().item()  # .item()是將張量轉換為Python數值
    total_loss += loss.item()

In [27]:
print(f"Test Loss: {total_loss / n_batches:.4f}, Test Accuracy: {total_acc / n_batches:.4f}")

Test Loss: 0.3559, Test Accuracy: 0.8434


In [35]:
model.eval()
input = "This movie was bad! I really don't like it."
encoding = tokenizer(
    input,
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
).to(device)   
logits = model(encoding["input_ids"], encoding["attention_mask"])
pred = (torch.sigmoid(logits) > 0.5).float().item()
print(f"Input: {input}")
print(f"Predicted label: {pred} (1=positive, 0=negative)")

Input: This movie was bad! I really don't like it.
Predicted label: 0.0 (1=positive, 0=negative)


In [39]:
batch = next(iter(train_loader))
print(batch.keys())


KeysView({'input_ids': tensor([[  101,   100,   100,  ...,     0,     0,     0],
        [  101,   100,  2145,  ..., 17771, 10029,   102],
        [  101,   100,   100,  ...,  2184,  1013,   102],
        ...,
        [  101,   100,   100,  ...,     0,     0,     0],
        [  101,   100,  2031,  ...,  2007,  1996,   102],
        [  101,  1008,  1008,  ...,  2062,  1997,   102]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]]), 'labels': tensor([0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1])})
